In [1]:
!pip install -q causal-learn pgmpy


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


# Phase 3: Structure Learning

Recover the CPDAG over filtered SAE features using the PC algorithm.

**Pipeline:**
1. Load prepared `(5000, 76)` data matrix.
2. Run PC skeleton (Fisher-Z CI, depth ≤ 3) — load from disk if cached.
3. Orient v-structures using causal-learn's `UCSepset.uc_sepset`.
4. Apply Meek's rules for propagation.
5. Save the directed PDAG.

**Note:** we use pgmpy for the skeleton phase (it has a depth cap parameter) and causal-learn for orientation (it correctly preserves edges that aren't v-structures, unlike pgmpy's `orient_colliders` which removes within-layer edges when temporal_ordering is supplied). See `notes.md` for the bug discussion.

In [2]:
import os

PROJECT_DIR = "data/pgm-final"
RESULTS_DIR = f"{PROJECT_DIR}/results"

os.makedirs(RESULTS_DIR, exist_ok=True)
print("Project dir:", PROJECT_DIR)
print("Results dir:", RESULTS_DIR)

Project dir: data/pgm-final
Results dir: data/pgm-final/results


In [3]:
import numpy as np
import json

X_continuous = np.load(f"{PROJECT_DIR}/activations/X_continuous.npy")
X_binary     = np.load(f"{PROJECT_DIR}/activations/X_binary.npy")
indices_L1   = np.load(f"{PROJECT_DIR}/activations/indices_L1.npy")
indices_L2   = np.load(f"{PROJECT_DIR}/activations/indices_L2.npy")

with open(f"{PROJECT_DIR}/activations/metadata.json", "r") as f:
    metadata = json.load(f)

N_L1 = metadata["n_features_L1"]
N_L2 = metadata["n_features_L2"]
N_TOTAL = metadata["n_features_total"]

print(f"Continuous: {X_continuous.shape}")
print(f"Binary:     {X_binary.shape}")
print(f"L1 features: {N_L1}, L2 features: {N_L2}, total: {N_TOTAL}")

Continuous: (5000, 76)
Binary:     (5000, 76)
L1 features: 30, L2 features: 46, total: 76


In [4]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from pgmpy.estimators import PC
import pandas as pd
import time
import pickle

df_continuous = pd.DataFrame(X_continuous)
est = PC(data=df_continuous)

/Users/siddharthraj/Documents/my-projects/pgm-final/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
PICKLE_PATH = f"{RESULTS_DIR}/pc_fisherz_alpha05_depth3.pkl"

if os.path.exists(PICKLE_PATH):
    print(f"Loading skeleton from cache: {PICKLE_PATH}")
    with open(PICKLE_PATH, "rb") as f:
        pc_result = pickle.load(f)
    skeleton = pc_result["skeleton"]
    sep_sets = pc_result["sep_sets"]
    elapsed  = pc_result["time"]
    print(f"  Edges: {skeleton.number_of_edges()}")
    print(f"  Original runtime: {elapsed:.1f}s (skipped recomputation)")
else:
    print("No cache — running PC skeleton...")
    t0 = time.time()
    skeleton, sep_sets = est.build_skeleton(
        ci_test="pearsonr",
        significance_level=0.05,
        max_cond_vars=3,
        show_progress=True,
    )
    elapsed = time.time() - t0
    pc_result = {
        "skeleton": skeleton,
        "sep_sets": sep_sets,
        "time": elapsed,
        "alpha": 0.05,
        "max_cond_vars": 3,
        "ci_test": "pearsonr",
    }
    with open(PICKLE_PATH, "wb") as f:
        pickle.dump(pc_result, f)
    print(f"  Edges: {skeleton.number_of_edges()}")
    print(f"  Runtime: {elapsed:.1f}s")
    print(f"  Saved to {PICKLE_PATH}")

Loading skeleton from cache: data/pgm-final/results/pc_fisherz_alpha05_depth3.pkl
  Edges: 382
  Original runtime: 1395.7s (skipped recomputation)


In [6]:
edges_arr = np.array([(min(e), max(e)) for e in skeleton.edges()])

n_L1_L1 = sum(1 for (i, j) in edges_arr if j < N_L1)
n_L2_L2 = sum(1 for (i, j) in edges_arr if i >= N_L1)
n_L1_L2 = sum(1 for (i, j) in edges_arr if i < N_L1 <= j)

print(f"Total skeleton edges: {len(edges_arr)}")
print(f"  L1—L1 (within layer 1):  {n_L1_L1:3d}  / max possible {N_L1*(N_L1-1)//2}")
print(f"  L2—L2 (within layer 2):  {n_L2_L2:3d}  / max possible {N_L2*(N_L2-1)//2}")
print(f"  L1—L2 (cross-layer):     {n_L1_L2:3d}  / max possible {N_L1*N_L2}")

Total skeleton edges: 382
  L1—L1 (within layer 1):   64  / max possible 435
  L2—L2 (within layer 2):  142  / max possible 1035
  L1—L2 (cross-layer):     176  / max possible 1380


## Orientation Phase (causal-learn)

`pgmpy` provides `orient_colliders`, but its implementation removes within-layer edges that participate in v-structures, especially when `temporal_ordering` is supplied. This dropped 201/382 of our skeleton edges in earlier testing, all within-layer.

We instead convert the pgmpy skeleton + separating sets into causal-learn's `CausalGraph` format and use `UCSepset.uc_sepset` for v-structure orientation followed by `Meek.meek` for propagation. These are the canonical algorithms from Spirtes/Glymour/Scheines and Meek 1995.

In [7]:
from causallearn.graph.GraphClass import CausalGraph

cg = CausalGraph(no_of_var=N_TOTAL)
cg.G.graph = np.zeros((N_TOTAL, N_TOTAL), dtype=np.int32)

# Encode each undirected edge as graph[i,j] = graph[j,i] = -1
for (i, j) in skeleton.edges():
    i, j = int(i), int(j)
    cg.G.graph[i, j] = -1
    cg.G.graph[j, i] = -1

n_undirected_in_cg = (cg.G.graph == -1).sum() // 2
print(f"Edges in CausalGraph: {n_undirected_in_cg}")
print(f"Expected from skeleton: {skeleton.number_of_edges()}")
assert n_undirected_in_cg == skeleton.number_of_edges(), "Edge count mismatch"

Edges in CausalGraph: 382
Expected from skeleton: 382


In [8]:
# causal-learn's UCSepset reads from cg.sepset[i, j], expecting either
# None (no sepset known) or a list of (sepset_tuple, p_value) entries.

cg.sepset = np.empty((N_TOTAL, N_TOTAL), dtype=object)
for i in range(N_TOTAL):
    for j in range(N_TOTAL):
        cg.sepset[i, j] = None

for pair_key, sep_set in sep_sets.items():
    nodes = list(pair_key)
    if len(nodes) != 2:
        continue  # skip degenerate keys if any
    i, j = int(nodes[0]), int(nodes[1])
    entry = [(tuple(int(x) for x in sep_set), 0.5)]
    cg.sepset[i, j] = entry
    cg.sepset[j, i] = entry

# Sanity check on one pair
sample_pair = list(sep_sets.keys())[0]
nodes = list(sample_pair)
i, j = int(nodes[0]), int(nodes[1])
print(f"Sample pair {sample_pair}:")
print(f"  pgmpy sep_set:  {sep_sets[sample_pair]}")
print(f"  cg.sepset[{i},{j}]: {cg.sepset[i, j]}")

Sample pair frozenset({np.int64(0), np.int64(1)}):
  pgmpy sep_set:  ()
  cg.sepset[0,1]: [((), 0.5)]


In [9]:
# Attach separating sets in the format causal-learn expects:
# cg.sepset[i, j] is either None, or a list of tuples of node indices.
# Each tuple is one sepset that made i, j conditionally independent.

cg.sepset = np.empty((N_TOTAL, N_TOTAL), dtype=object)
# Leave as None where no sepset is recorded.

for pair_key, sep_set in sep_sets.items():
    nodes = list(pair_key)
    if len(nodes) != 2:
        continue
    i, j = int(nodes[0]), int(nodes[1])
    sepset_tuple = tuple(int(x) for x in sep_set)
    cg.sepset[i, j] = [sepset_tuple]
    cg.sepset[j, i] = [sepset_tuple]

# Sanity check
sample_pair = list(sep_sets.keys())[0]
nodes = list(sample_pair)
i, j = int(nodes[0]), int(nodes[1])
print(f"Sample pair {sample_pair}:")
print(f"  pgmpy sep_set:  {sep_sets[sample_pair]}")
print(f"  cg.sepset[{i},{j}]: {cg.sepset[i, j]}")

# Count how many pairs have a sepset
n_with_sepset = sum(1 for i in range(N_TOTAL) for j in range(N_TOTAL) if cg.sepset[i, j] is not None)
print(f"\nTotal (i,j) entries with a sepset: {n_with_sepset}")
print(f"  (each unordered pair counted twice = {n_with_sepset // 2} unique pairs)")

Sample pair frozenset({np.int64(0), np.int64(1)}):
  pgmpy sep_set:  ()
  cg.sepset[0,1]: [()]

Total (i,j) entries with a sepset: 4936
  (each unordered pair counted twice = 2468 unique pairs)


In [11]:
from causallearn.utils.PCUtils import UCSepset, Meek

t0 = time.time()
cg_v = UCSepset.uc_sepset(cg, priority=0)
t_v = time.time() - t0
print(f"V-structures oriented in {t_v:.3f}s")

t0 = time.time()
cg_meek = Meek.meek(cg_v)
t_m = time.time() - t0
print(f"Meek's rules applied in {t_m:.3f}s")

V-structures oriented in 19.450s
Meek's rules applied in 0.215s


In [12]:
# Causal-learn convention:
#   graph[i,j] = -1, graph[j,i] = -1   →  i — j  (undirected)
#   graph[i,j] = -1, graph[j,i] =  1   →  i → j
#   graph[i,j] =  1, graph[j,i] = -1   →  j → i

G = cg_meek.G.graph

directed_edges   = []     # (src, dst) pairs
undirected_edges = []     # (i, j) with i < j

for i in range(N_TOTAL):
    for j in range(i+1, N_TOTAL):
        a, b = G[i, j], G[j, i]
        if a == -1 and b == -1:
            undirected_edges.append((i, j))
        elif a == -1 and b == 1:
            directed_edges.append((i, j))
        elif a == 1 and b == -1:
            directed_edges.append((j, i))

print(f"Directed:   {len(directed_edges)}")
print(f"Undirected: {len(undirected_edges)}")
print(f"Total:      {len(directed_edges) + len(undirected_edges)}")
print(f"Skeleton was: {skeleton.number_of_edges()}")

# Layer breakdown
def layer_of(x): return "L1" if x < N_L1 else "L2"

dir_L1_L2 = sum(1 for (s, d) in directed_edges if s < N_L1 and d >= N_L1)
dir_L2_L1 = sum(1 for (s, d) in directed_edges if s >= N_L1 and d < N_L1)
dir_L1_L1 = sum(1 for (s, d) in directed_edges if s < N_L1 and d < N_L1)
dir_L2_L2 = sum(1 for (s, d) in directed_edges if s >= N_L1 and d >= N_L1)

und_L1_L2 = sum(1 for (i, j) in undirected_edges if i < N_L1 <= j)
und_L1_L1 = sum(1 for (i, j) in undirected_edges if j < N_L1)
und_L2_L2 = sum(1 for (i, j) in undirected_edges if i >= N_L1)

print()
print("DIRECTED edges by category:")
print(f"  L1 → L2 (expected dominant):  {dir_L1_L2}")
print(f"  L2 → L1 (should be 0):        {dir_L2_L1}")
print(f"  L1 → L1 (within layer 1):     {dir_L1_L1}")
print(f"  L2 → L2 (within layer 2):     {dir_L2_L2}")
print()
print("UNDIRECTED edges by category:")
print(f"  L1 — L2 (cross-layer):    {und_L1_L2}")
print(f"  L1 — L1 (within layer 1): {und_L1_L1}")
print(f"  L2 — L2 (within layer 2): {und_L2_L2}")

Directed:   382
Undirected: 0
Total:      382
Skeleton was: 382

DIRECTED edges by category:
  L1 → L2 (expected dominant):  176
  L2 → L1 (should be 0):        0
  L1 → L1 (within layer 1):     64
  L2 → L2 (within layer 2):     142

UNDIRECTED edges by category:
  L1 — L2 (cross-layer):    0
  L1 — L1 (within layer 1): 0
  L2 — L2 (within layer 2): 0


In [13]:
ORIENTED_PATH = f"{RESULTS_DIR}/pc_fisherz_alpha05_depth3_oriented.pkl"

oriented_result = {
    "directed_edges":   directed_edges,
    "undirected_edges": undirected_edges,
    "cg_meek_graph":    cg_meek.G.graph,
    "indices_L1":       indices_L1,
    "indices_L2":       indices_L2,
    "N_L1": N_L1,
    "N_L2": N_L2,
    "alpha": 0.05,
    "max_cond_vars": 3,
    "ci_test": "pearsonr",
    "skeleton_runtime_s": elapsed,
    "orientation_runtime_s": t_v + t_m,
}

with open(ORIENTED_PATH, "wb") as f:
    pickle.dump(oriented_result, f)

print(f"Saved oriented PDAG to: {ORIENTED_PATH}")
print(f"  {len(directed_edges)} directed + {len(undirected_edges)} undirected edges")

Saved oriented PDAG to: data/pgm-final/results/pc_fisherz_alpha05_depth3_oriented.pkl
  382 directed + 0 undirected edges


In [14]:
G = cg_meek.G.graph

# Look at a few entries to see what values appear
unique_pairs = set()
for i in range(N_TOTAL):
    for j in range(i+1, N_TOTAL):
        a, b = G[i, j], G[j, i]
        unique_pairs.add((int(a), int(b)))

print("All unique (graph[i,j], graph[j,i]) pairs that appear:")
for p in sorted(unique_pairs):
    count = sum(1 for i in range(N_TOTAL) for j in range(i+1, N_TOTAL)
                if G[i, j] == p[0] and G[j, i] == p[1])
    print(f"  {p}: {count} pairs")

All unique (graph[i,j], graph[j,i]) pairs that appear:
  (-1, 1): 382 pairs
  (0, 0): 2468 pairs
